In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from dotenv import load_dotenv
import os
from pathlib import Path
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm
import gc

In [7]:
load_dotenv()

True

In [8]:
DATA_DIR = Path(os.environ["DATA_DIR"])

IMAGE_DIR = DATA_DIR / "raw"
OUTPUT_CSV = DATA_DIR / "labels.csv"

IMAGE_EXTENSIONS = [".jpg"]

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded successfully.


In [10]:
PROMPT = """
You are an expert marketing image analyst.

Analyze the image.

Choose exactly ONE mood:

- Energetic
- Calm
- Professional
- Playful

Choose exactly ONE content type:

- Product Showcase
- Lifestyle
- Testimonial
- Promotional

Return ONLY valid JSON.

{
    "mood": "Professional",
    "content_type": "Product Showcase"
}
"""

In [11]:
def analyze_image(image_path):
    image = Image.open(image_path).convert("RGB")

    # Resize if too large
    MAX_RES = 1024
    if image.width > MAX_RES or image.height > MAX_RES:
        image.thumbnail((MAX_RES, MAX_RES))

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)  

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = generated_ids[:, input_length:]
    # print(f"Input length: {input_length}, Generated IDs: {generated_ids.shape[1]}")

    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    
    del inputs, generated_ids
    torch.cuda.empty_cache()
    gc.collect()
    
    return output_text

In [12]:
# result  = analyze_image(IMAGE_DIR / "0_0a7ad22c5b040c00.jpg")
# result = result[0].strip()
# start = result.find("{")
# end = result.rfind("}") + 1
# print(json.loads(result[start:end]))

In [13]:
jpg_files = sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(jpg_files)} images.\n")

Found 1632 images.



In [14]:
rows  = []

for jpg_file in tqdm(jpg_files, desc="Processing images"):
    result = analyze_image(jpg_file)
    result = result[0].strip()
    start = result.find("{")
    end = result.rfind("}") + 1
    try:
        result_json = json.loads(result[start:end])
        rows.append({
            "image_name": jpg_file.name,
            "mood": result_json.get("mood", ""),
            "content_type": result_json.get("content_type", ""),
        })
    except json.JSONDecodeError:
        print(f"Failed to parse JSON for {jpg_file.name}: {result}")
    

df = pd.DataFrame(rows)
print(f"Saved {len(df)} rows")

Processing images:   3%|▎         | 48/1632 [02:56<1:42:27,  3.88s/it]

Failed to parse JSON for 0_0936f36a5a97b898.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images:   3%|▎         | 51/1632 [03:07<1:39:11,  3.76s/it]

Failed to parse JSON for 0_0a51f3b1110e3b8f.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images:   5%|▌         | 84/1632 [05:07<1:39:04,  3.84s/it]

...

Failed to parse JSON for 3_48312460bff0993c.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images:  95%|█████████▍| 1549/1632 [1:33:50<04:59,  3.60s/it]

Failed to parse JSON for 3_8cca1b09cdc0239e.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images: 100%|██████████| 1632/1632 [1:38:42<00:00,  3.63s/it]

Saved 1617 rows


In [15]:
df["mood"].value_counts()

,count
mood,
Calm,604
Professional,504
Energetic,319
Playful,190


In [16]:
df["content_type"].value_counts()

,count
content_type,
Lifestyle,860
Promotional,506
Product Showcase,239
Testimonial,7
Sports,4
Art,1


In [17]:
valid_types = ['Lifestyle', 'Promotional', 'Product Showcase']
invalid_rows = df[~df['content_type'].isin(valid_types)]
print(invalid_rows[['image_name', 'content_type']])

                  image_name content_type
68    0_15889a73635f10ba.jpg  Testimonial
110   0_332869175a4a14a0.jpg  Testimonial
343   0_a8db7ceafb5a2df3.jpg  Testimonial
438   0_d066d8c21f939429.jpg       Sports
758   1_89d9552a610e6a67.jpg       Sports
859   1_d066d8c21f939429.jpg       Sports
932   2_038399f2c7261a42.jpg       Sports
1026  2_332869175a4a14a0.jpg  Testimonial
1143  2_7a317e36fa5bdffe.jpg  Testimonial
1205  2_959eeeca82621137.jpg  Testimonial
1336  2_d86621ad69bb5f7b.jpg  Testimonial
1367  2_ebf835d2eab7770f.jpg          Art


In [ ]:
df.loc[df['content_type'] == 'Testimonial', 'content_type'] = 'Lifestyle'
df.loc[df['content_type'] == 'Sports', 'content_type'] = 'Lifestyle'
df.loc[df['content_type'] == 'Art', 'content_type'] = 'Promotional'

In [19]:
empty_rows = df[df['mood'].isna() | (df['mood'] == '') | (df['mood'] == 'Unknown')]
print(empty_rows[['image_name', 'mood', 'content_type']])

Empty DataFrame
Columns: [image_name, mood, content_type]
Index: []


In [20]:
df = df.drop_duplicates(subset=['image_name'], keep='first')

In [21]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"{len(df)} rows saved")
df.head() 

1617 rows saved


,image_name,mood,content_type
0,0_00124ea83b50f37d.jpg,Calm,Promotional
1,0_005617151e189073.jpg,Professional,Product Showcase
2,0_00e26c825cbbf94f.jpg,Energetic,Product Showcase
3,0_00e98cc91b811366.jpg,Calm,Lifestyle
4,0_00f3733aaa881434.jpg,Energetic,Lifestyle
